# 09. Creating a "Manual RAG" Prompt Agent

This script creates another new Foundry agent, `"cloudxeus-support-rag-agent"`, but notice what's **absent**
from its `PromptAgentDefinition`: there's no `tools=[...]` at all — no `FileSearchTool`, no
`AzureAISearchTool`. Instead, the agent's `instructions` (system prompt) are written to expect that the
**caller** will supply retrieved source text directly inside the user message at query time, and the agent's
job is purely to answer strictly from whatever sources it's given, citing them, and refusing to invent
anything not present in those sources. This is the **"manual RAG" / prompt-stuffing** pattern: retrieval
happens entirely in your own client code (see `10_customer_rag_client.py`, which builds on `08_ai_search.py`'s
search technique), and the agent never touches the search index itself.

**Difficulty:** Intermediate


## Prerequisites

**pip3 packages:** `azure-ai-projects`, `azure-identity`, `python-dotenv` — already in root `requirements.txt`.

**Azure resource(s) required:** A Foundry project with a deployed chat model. (The Azure AI Search resource
used later, in `10_customer_rag_client.py`, is not needed just to *create* this agent — only to query it with
real grounding data.)

**Environment variables** (root `.env`):
- `AZURE_AI_PROJECT_ENDPOINT`
- `AZURE_AI_MODEL_DEPLOYMENT`
- `AZURE_AI_RAG_AGENT_NAME` (optional, defaults to `"cloudxeus-support-rag-agent"`)


## What You'll Learn

- The "manual RAG" pattern: an agent with a strict, source-citing system prompt but **no retrieval tool** —
  retrieval and prompt-construction are entirely the caller's responsibility
- How a well-written system prompt can enforce grounding discipline (answer only from provided sources, cite
  them, and refuse to invent facts) even without a built-in retrieval mechanism
- `AzureCliCredential` as an alternative Entra ID credential to `DefaultAzureCredential`


### Setup and creating the agent

The system prompt (`SYSTEM_PROMPT`) is the entire grounding mechanism here — it explicitly instructs the model
to answer **only** from sources it's given, cite the source URL, admit when the knowledge base doesn't have an
answer, and never invent policies/prices/refund rules. This only works because `10_customer_rag_client.py`
will always inject retrieved source text into the *user* message before calling this agent — the agent has no
independent way to fetch data itself.

Note the credential here is `AzureCliCredential()`, not `DefaultAzureCredential()` as in earlier scripts.

- **💡 Exam tip:** AI-102 distinguishes **"bring your own retrieval"** RAG (this pattern — you query a search
  index yourself and stuff results into the prompt) from **"on your data" / tool-based RAG**, where Foundry's
  `AzureAISearchTool` or `FileSearchTool` perform retrieval automatically as part of the agent's execution.
  Both are valid AI-102 exam topics, and recognizing which one a piece of code implements matters.
- **🔄 Alternatives:** `AzureCliCredential` only tries your local `az login` session (fails fast and
  predictably if you're not logged in), whereas `DefaultAzureCredential` tries a whole chain of credential
  sources (environment variables, managed identity, Visual Studio, Azure CLI, etc.) — useful in code that must
  run unmodified in more environments, but slightly slower/less predictable to debug locally.


In [ ]:
import os

from dotenv import load_dotenv
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

load_dotenv()

PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
DEPLOYMENT_NAME = os.getenv("AZURE_AI_MODEL_DEPLOYMENT", "gpt-5.4")
RAG_AGENT_NAME = os.getenv("AZURE_AI_RAG_AGENT_NAME", "cloudxeus-support-rag-agent")

SYSTEM_PROMPT = """
You are a customer support assistant for CloudXeus Technology Services.

Answer the customer's question using ONLY the provided sources.

After your answer, cite the source URL you used.

If the sources do not contain the answer, say:
"I don't have that information in the available knowledge base."

Then suggest contacting support@cloudxeus.com.

Never invent policies, prices, refund rules, or timelines.
"""

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=AzureCliCredential()
)

agent = project.agents.create_version(
    agent_name=RAG_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=DEPLOYMENT_NAME,
        instructions=SYSTEM_PROMPT,
    ),
)

print("Agent created:")
print(f"  ID      : {agent.id}")
print(f"  Name    : {agent.name}")
print(f"  Version : {agent.version}")


## Summary

This notebook creates `cloudxeus-support-rag-agent`, a Foundry prompt agent whose entire grounding contract is
enforced through its system prompt rather than any attached tool. It's ready to be invoked, but by itself it
has no data to answer with — `10_customer_rag_client.py` is what actually retrieves relevant document chunks
from Azure AI Search and constructs the "only from these sources" prompt this agent expects.


## Try It Yourself

1. **Easy:** Invoke this agent right now (before running `10_customer_rag_client.py`) with a plain question
   and no sources — confirm it responds with something close to the "I don't have that information..."
   fallback, since it was given no grounding text.
2. **Intermediate:** Tighten `SYSTEM_PROMPT` further — e.g. require the answer to be under 3 sentences, or
   require citing every source used, not just one.
3. **Advanced:** Compare this "manual RAG" approach against attaching a `FileSearchTool` (see
   `11_azure_ai_foundry/07_knowledge_rag`) to a similarly-instructed agent — which approach gives you more
   control over the exact retrieval query and prompt construction, and which is less code to maintain?
